# Ajuste QLoRA de Qwen2.5-3B al tono de Magister

Colab con GPU T4. Sube `sft-entrenamiento.jsonl` y `sft-validacion.jsonl` a `Drive/magister/datos/`. Entrena solo sobre la respuesta y guarda el adaptador en Drive.

In [ ]:
!pip install -q unsloth
from google.colab import drive
drive.mount('/content/drive')
RUTA = '/content/drive/MyDrive/magister'

In [ ]:
from unsloth import FastLanguageModel
modelo, tok = FastLanguageModel.from_pretrained('unsloth/Qwen2.5-3B-Instruct', max_seq_length=2048, load_in_4bit=True)
modelo = FastLanguageModel.get_peft_model(modelo, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], random_state=7)

In [ ]:
from datasets import load_dataset
datos = load_dataset('json', data_files={'train': f'{RUTA}/datos/sft-entrenamiento.jsonl', 'validation': f'{RUTA}/datos/sft-validacion.jsonl'})
datos = datos.map(lambda e: {'text': tok.apply_chat_template(e['messages'], tokenize=False)})

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
entrenador = SFTTrainer(model=modelo, tokenizer=tok, train_dataset=datos['train'], eval_dataset=datos['validation'],
    args=SFTConfig(output_dir=f'{RUTA}/salida', per_device_train_batch_size=2, gradient_accumulation_steps=8, num_train_epochs=2,
        learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type='cosine', eval_strategy='steps', eval_steps=0.25, seed=7,
        dataset_text_field='text'))
entrenador = train_on_responses_only(entrenador, instruction_part='<|im_start|>user
', response_part='<|im_start|>assistant
')
historia = entrenador.train()

In [ ]:
modelo.save_pretrained(f'{RUTA}/adaptador'); tok.save_pretrained(f'{RUTA}/adaptador')
modelo.save_pretrained_gguf(f'{RUTA}/gguf', tok, quantization_method='q4_k_m')